In [10]:
import os
import cv2
import sys
import numpy as np
sys.path.append('practicum_work/src')
from dataset.eda import missed_labels, class_distribution, size_distribution
from dataset.coco import generate_coco_dataset
from dataset.label import generate_labeled_dataset

def find_train_unlabeled(folder: str, img_dir: str, lbl_dir: str):
    img_dir = os.path.join(img_dir, folder)
    label_dir = os.path.join(lbl_dir, folder)
    imgs = os.listdir(img_dir)
    for i in range(len(imgs)):
        file_name = os.path.splitext(imgs[i])[0] + '.png'
        label_path = os.path.join(label_dir, file_name)
        if not os.path.exists(label_path):
            print('File {} does not exist.'.format(label_path))

base_img_dir = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\base\img'
base_lbl_dir = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\base\labels'

for folder in ['train', 'test', 'val']:
    find_train_unlabeled(folder, base_img_dir, base_lbl_dir)

ModuleNotFoundError: No module named 'dataset.eda'

In [6]:
base_dir = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\base'
output_root = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\labeled'
for split in ['train', 'val', 'test']:
    img_path = os.path.join(base_dir, 'img', split)
    lbl_path = os.path.join(base_dir, 'labels', split)
    output_path = os.path.join(output_root, split)
    generate_labeled_dataset(img_path, lbl_path, output_path, {1: [255, 0, 0], 2: [0, 255, 0]})

NameError: name 'generate_labeled_dataset' is not defined

# Exploratory Data Analysis (EDA)

In [ ]:
categories = ['background', 'cat', 'dog']
img_path = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\base\img\train'
lbl_path = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\base\labels\train'

missed_labels(img_path, lbl_path)
class_distribution(lbl_path, categories=categories)
size_distribution(img_path)

# COCO Annotations Preparation

In [ ]:
output_folder = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\coco'
base_dir = r'C:\Users\igors\Projects\mmsegmentation-benchmarking\practicum_work\dataset\base'

for split in ['train', 'val', 'test']:
    img_split = os.path.join(base_dir, 'img', split)
    lbl_split = os.path.join(base_dir, 'labels', split)
    output_split_folder = os.path.join(output_folder, split)
    os.makedirs(output_split_folder, exist_ok=True)
    output_json = os.path.join(output_split_folder, 'annotations.json')
    generate_coco_dataset(img_split, lbl_split, output_json, categories=['background', 'class1', 'class2'])

# Experiment 1: DeepLabV3+ ResNet-50 D8

In [ ]:
# !python tools/train.py \
#     configs/experiments/deeplabv3plus_r50_d8_20k_coco_animals.py \
#     --work-dir ./practicum_work/experiments/deeplabv3plus_r50_d8 \
#     --amp

In [ ]:
!python tools/test.py \
    configs/experiments/deeplabv3plus_r50_d8_20k_coco_animals.py \
    practicum_work/experiments/deeplabv3plus_r50_d8/iter_20000.pth

# Experiment 2: DeepLabV3+ ResNet-50 D16

In [ ]:
# !python tools/train.py \
#     configs/experiments/deeplabv3plus_r50_d16_20k_coco_animals.py \
#     --work-dir ./practicum_work/experiments/deeplabv3plus_r50_d16 \
#     --amp

In [ ]:
!python tools/test.py \
    configs/experiments/deeplabv3plus_r50_d16_20k_coco_animals.py \
    practicum_work/experiments/deeplabv3plus_r50_d16/iter_20000.pth

# Experiment 3: DeepLabV3+ ResNet-101 D16

In [ ]:
# !python tools/train.py \
#     configs/experiments/deeplabv3plus_r101_d16_20k_coco_animals.py \
#     --work-dir ./practicum_work/experiments/deeplabv3plus_r101_d16 \
#     --amp

In [ ]:
!python tools/test.py \
    configs/experiments/deeplabv3plus_r101_d16_20k_coco_animals.py \
    practicum_work/experiments/deeplabv3plus_r101_d16/iter_20000.pth

# Results Analysis: Best DeepLabv3+ - ResNet50 - d8

In [ ]:
!python practicum_work/src/analysis/dump_model_predictions.py \
    configs/experiments/deeplabv3plus_r50_d8_20k_coco_animals.py \
    practicum_work/experiments/deeplabv3plus_r50_d8/iter_20000.pth \
    practicum_work/dataset/base/img/test \
    practicum_work/experiments/deeplabv3plus_r50_d8/preds_test

In [ ]:
!python practicum_work/src/analysis/save_best_on_worst_based_on_individual_dice_score.py \
    practicum_work/experiments/deeplabv3plus_r50_d8/preds_test \
    practicum_work/dataset/base/labels/test \
    practicum_work/dataset/base/img/test \
    practicum_work/experiments/deeplabv3plus_r50_d8/best_worst_test \
    --num-classes 3

In [ ]:
import optuna
import pandas as pd
from IPython.display import display

from mmengine.config import Config
from mmengine.runner import Runner
from mmengine.hooks import Hook
from mmseg.registry import HOOKS

# ==========================================
# 1. ОПРЕДЕЛЯЕМ ХУК БЕЗ РЕГИСТРАЦИИ В CONFIG
# ==========================================
class OptunaPruningHook(Hook):
    def __init__(self, trial, metric='val/mDice', interval=1):
        self.trial = trial
        self.metric = metric

    def after_val_epoch(self, runner, metrics=None):
        # Забираем метрику для Optuna
        if metrics is not None and self.metric in metrics:
            current_score = metrics[self.metric]
        else:
            try:
                current_score = runner.message_hub.get_scalar(self.metric).current()
            except:
                return # Если метрики еще нет, пропускаем

        step = runner.epoch if runner.train_loop.by_epoch else runner.iter
        self.trial.report(current_score, step)

        # Если эксперимент идет хуже других — обрубаем его
        if self.trial.should_prune():
            raise optuna.TrialPruned(f"Trial {self.trial.number} pruned at step {step}")


# ==========================================
# 2. ФУНКЦИЯ ПОДБОРА ПАРАМЕТРОВ
# ==========================================
def objective(trial):
    # --- Выбор архитектуры из 5 доступных ---
    arch_config = trial.suggest_categorical('config', [
        'configs/experiments/deeplabv3plus_r50_d8_20k_coco_animals.py',
        'configs/experiments/pspnet_r50_d8_20k_coco_animals.py',
        'configs/experiments/fcn_r50_d8_20k_coco_animals.py',
        'configs/experiments/upernet_r50_20k_coco_animals.py',
        'configs/experiments/segformer_mit-b0_20k_coco_animals.py'
    ])
    
    # --- Подбор гиперпараметров ---
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    prob_flip = trial.suggest_float('prob_flip', 0.0, 1.0)
    
    # Загружаем конфиг
    cfg = Config.fromfile(arch_config)
    cfg.optim_wrapper.optimizer.lr = lr
    cfg.optim_wrapper.optimizer.weight_decay = weight_decay
    
    # Меняем вероятность RandomFlip
    if hasattr(cfg, 'train_pipeline'):
        for transform in cfg.train_pipeline:
            if transform.get('type') == 'RandomFlip':
                transform['prob'] = prob_flip
    
    # Указываем директорию триала
    cfg.work_dir = f'./work_dirs/optuna_study/trial_{trial.number}'
    
    # Настройки для ускорения перебора
    cfg.train_cfg.max_iters = 6000
    cfg.train_cfg.val_interval = 2000
    cfg.default_hooks.checkpoint.interval = 6000
    
    # === ОБУЧАЕМ И ПОЛУЧАЕМ ОЦЕНКУ ===
    try:
        runner = Runner.from_cfg(cfg)
        
        # РЕГИСТРИРУЕМ ХУК ВНУТРЬ РАННЕРА
        # Это позволяет передать объект trial напрямую, не ломая конфиг
        optuna_hook = OptunaPruningHook(trial=trial, metric='val/mDice')
        runner.register_hook(optuna_hook, priority='LOWEST')
        
        runner.train()
        
        # Получаем финальный mDice
        val_metrics = runner.val_loop.run()
        return val_metrics['mDice']
        
    except optuna.TrialPruned as e:
        # Успешно обрубили плохой эксперимент
        raise e
    except Exception as e:
        print(f"Эксперимент №{trial.number} сломался с ошибкой: {e}")
        return 0.0


# ==========================================
# 3. ЗАПУСК ИССЛЕДОВАНИЯ
# ==========================================
study = optuna.create_study(
    study_name='mmseg_benchmark_tuner',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2000)
)

print("Начинаю перебор гиперпараметров для 5 архитектур (15 попыток)...")
study.optimize(objective, n_trials=15)

# ==========================================
# 4. ВЫВОД РЕЗУЛЬТАТОВ
# ==========================================
print("\n🏆 ЛУЧШИЕ ПАРАМЕТРЫ:")
print(study.best_params)

df = study.trials_dataframe()
cols = ['number', 'value', 'state', 'params_config', 'params_lr', 'params_prob_flip', 'params_weight_decay']
df = df[cols].sort_values(by='value', ascending=False)
display(df)
